# Intel Image Classification — end-to-end walkthroughSix scene categories — **buildings, forest, glacier, mountain, sea, street** — classified byfine-tuning an ImageNet-pretrained convolutional network.The story this notebook tells, in order:1. **Look at the data first.** Class balance, image sizes, and a grid of actual photos.2. **Why transfer learning.** What we inherit from ImageNet and what we replace.3. **Train** in two stages — frozen backbone, then full fine-tune.4. **Read the results honestly** — not just accuracy, but *which* classes fail and *how*.5. **Look at the mistakes** and at what the model was attending to (Grad-CAM).Every heavy step reuses the same code as the command-line scripts, so nothing hereis a notebook-only reimplementation that can drift out of sync.

## 0. SetupIf you have not downloaded the dataset yet:```bashpython scripts/download_data.py```No Kaggle token? Generate a stand-in dataset so every cell below still runs:```bashpython scripts/make_sample_data.py --dest data/sample```…then set `DATA_ROOT = "data/sample"` in the next cell.

In [ ]:
import sys, warningsfrom pathlib import PathREPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(REPO_ROOT / "src"))warnings.filterwarnings("ignore", category=UserWarning)import numpy as npimport torchfrom ic.config import load_configfrom ic.utils import resolve_device, set_seed, get_logger, make_run_dir, count_parametersfrom ic.viz import use_styleuse_style()DATA_ROOT = "data/intel"        # <- "data/sample" if you generated the synthetic setEPOCHS    = 8                   # <- 2 is plenty just to see the machinery workMODEL     = "resnet18"          # resnet18 | resnet34 | resnet50 | efficientnet_b0 | ...cfg = load_config(    REPO_ROOT / "configs" / "default.yaml",    [f"data.root={REPO_ROOT / DATA_ROOT}", f"train.epochs={EPOCHS}", f"model.name={MODEL}"],)set_seed(cfg.seed)device = resolve_device(cfg.device)print(f"torch {torch.__version__} · device {device}")

## 1. Look at the data before touching a modelTwo questions decide most of what follows: **is it balanced?** (if not, we need class weights orbalanced sampling) and **what do the images actually look like?** (which tells us whataugmentation is safe — flipping a landscape horizontally is fine; flipping a digit is not).

In [ ]:
from ic.data import build_datasets, build_dataloaders, class_distributionfrom ic.viz import plot_class_distributiondatasets = build_datasets(cfg)classes = datasets["classes"]print(f"classes  : {classes}")print(f"train    : {len(datasets['train']):,}")print(f"val      : {len(datasets['val']):,}   (carved out of train, stratified)" if datasets['val'] is not None else "val      : none (data.val_split is 0)")print(f"test     : {len(datasets['test']):,}   (untouched until the very end)")distribution = class_distribution(datasets["train_targets"], classes)imbalance = max(distribution.values()) / min(distribution.values())print(f"\nimbalance ratio (largest class / smallest): {imbalance:.2f}x")plot_class_distribution(distribution, title="Training images per class");

An imbalance ratio near 1.0 means plain cross-entropy is fine. Past roughly 3x, turn on`train.class_weights` in the config — otherwise the model can score well by quietlyignoring the rare classes.

In [ ]:
from ic.viz import show_batchloaders, _ = build_dataloaders(cfg, datasets)images, labels = next(iter(loaders["train"]))print(f"batch tensor: {tuple(images.shape)}  (batch, channels, height, width)")show_batch(images, labels, classes, cfg.data.mean, cfg.data.std, n=12, cols=6);

These are the **augmented** training views — random crops, flips, colour jitter, and theoccasional erased patch. That is deliberate: the model should never see the exact sameimage twice, which is the cheapest regularisation available. The validation and testpipelines skip all of it and just resize + centre-crop.

## 2. Why transfer learningA ResNet trained on ImageNet already knows edges, textures, and object parts. Those earlylayers transfer to almost any natural-image task. What does *not* transfer is the final1000-way classifier, so we throw it away and bolt on a fresh 6-way head.Training then runs in two stages:| Stage | Backbone | Head | Learning rate | Why ||---|---|---|---|---|| 1 (epochs 1-2) | frozen | training | 1e-3 | Random head weights would otherwise send huge gradients through good features and wreck them || 2 (epoch 3+) | training | training | 1e-4 backbone / 1e-3 head | Once the head is sane, gently adapt the features to *our* six classes |

In [ ]:
from ic.model import build_modelmodel = build_model(    name=cfg.model.name,    num_classes=len(classes),    pretrained=True,    dropout=cfg.model.dropout,    freeze_backbone=cfg.model.freeze_backbone,).to(device)trainable, total = count_parameters(model)print(f"{cfg.model.name}: {trainable:,} trainable of {total:,} total "      f"({trainable / total:.1%} — the head only, while the backbone is frozen)")

## 3. Train`fit` handles the loop: warmup + cosine schedule, mixed precision on CUDA, gradientclipping, the stage-2 unfreeze, early stopping, and checkpointing the best validationaccuracy. On a GPU expect a couple of minutes per epoch; on CPU, considerably more —drop `EPOCHS` to 2 if you just want to watch it work.

In [ ]:
from ic.engine import fitfrom ic.viz import plot_historyrun_dir = make_run_dir(REPO_ROOT / "outputs", "notebook", cfg.model.name)logger = get_logger("ic.notebook", run_dir / "train.log")result = fit(model, loaders, cfg, device, classes, run_dir, logger,             train_targets=datasets["train_targets"])plot_history(result["history"]);

**How to read these curves.** Training and validation loss should fall together. If thevalidation curve turns upward while training keeps dropping, the model has startedmemorising — stop earlier, augment harder, or raise weight decay. A visible kink at theunfreeze epoch is expected and healthy: more parameters just came online.

## 4. Evaluate on the held-out test setAccuracy alone hides too much, so we also look at balanced accuracy (immune to classimbalance), macro F1 (every class counts equally), and the full per-class breakdown.

In [ ]:
from ic.engine import build_criterion, evaluatefrom ic.metrics import compute_metrics, format_summary, most_confused_pairsfrom ic.utils import load_checkpointfrom ic.viz import plot_confusion_matrix, plot_per_class_metriccheckpoint = load_checkpoint(result["best_checkpoint"], map_location=device)model.load_state_dict(checkpoint["state_dict"])criterion = build_criterion(cfg, None, len(classes), device)outputs = evaluate(model, loaders["test"], criterion, device, desc="test", return_outputs=True)metrics = compute_metrics(outputs["targets"], outputs["preds"], outputs["probs"], classes)print(format_summary(metrics))

In [ ]:
plot_confusion_matrix(metrics["confusion_matrix"], classes, title="Confusion matrix — test set");

In [ ]:
plot_per_class_metric([metrics["per_class"][c]["f1"] for c in classes], classes, "F1");print("Where the model actually struggles:")for true_name, pred_name, count in most_confused_pairs(metrics["confusion_matrix"], classes):    print(f"  {true_name:<12} -> {pred_name:<12} {count:>5,} images")

On this dataset the reliable pattern is **glacier <-> mountain**: snow-covered peaks belong toboth categories by any reasonable human standard, and a fair number of the labels arearguable. That confusion is a property of the dataset, not a bug in the model — worthsaying out loud in any write-up, because it caps how high accuracy can honestly go.

## 5. Look at the mistakesAggregate metrics tell you *how much* is wrong. Only the images tell you *why*.

In [ ]:
wrong = np.where(outputs["preds"] != outputs["targets"])[0]confidence = outputs["probs"][wrong, outputs["preds"][wrong]]most_confident_errors = wrong[np.argsort(confidence)[::-1][:12]]   # wrong *and* sure of itselftest_ds = datasets["test"]error_images = torch.stack([test_ds[i][0] for i in most_confident_errors])error_truth = torch.tensor([test_ds[i][1] for i in most_confident_errors])print(f"{len(wrong):,} of {len(outputs['targets']):,} test images misclassified "      f"({len(wrong) / len(outputs['targets']):.1%})")from ic.viz import show_batchshow_batch(error_images, error_truth, classes, cfg.data.mean, cfg.data.std,           preds=outputs["preds"][most_confident_errors], n=12, cols=6);

## 6. What was the model looking at?Grad-CAM weights the last convolutional feature maps by the gradient of the winning class,which gives a rough map of the pixels that drove the decision. It is a sanity check, notproof: if the heat sits on a watermark or a border instead of the subject, the model haslearned a shortcut.

In [ ]:
import matplotlib.pyplot as pltfrom ic.data import denormalizefrom ic.gradcam import GradCAM, overlay_heatmapsample_indices = [int(i) for i in np.random.default_rng(0).choice(len(test_ds), 4, replace=False)]fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))with GradCAM(model) as cam_fn:    for column, index in enumerate(sample_indices):        tensor, true_label = test_ds[index]        tensor = tensor.to(device)        cam = cam_fn(tensor.unsqueeze(0))        with torch.no_grad():            probs = torch.softmax(model(tensor.unsqueeze(0)).float(), 1)[0].cpu().numpy()        predicted = int(probs.argmax())        rgb = denormalize(tensor, cfg.data.mean, cfg.data.std).permute(1, 2, 0).numpy()        axes[0, column].imshow(rgb)        axes[0, column].set_title(f"true: {classes[true_label]}", fontsize=10)        axes[1, column].imshow(overlay_heatmap(rgb, cam))        axes[1, column].set_title(f"{classes[predicted]} · {probs[predicted]:.0%}", fontsize=10)        for row in (0, 1):            axes[row, column].axis("off")fig.suptitle("Grad-CAM — input above, attention below", x=0.01, ha="left", fontsize=13)fig.tight_layout();

## 7. Where to take it next- **A stronger backbone.** `MODEL = "efficientnet_b0"` or `"convnext_tiny"` usually buys a  point or two for a modest cost. Re-run this notebook; nothing else changes.- **Test-time augmentation.** Average predictions over the image and its mirror.- **Mixup / CutMix.** Helps most once plain augmentation has stopped paying.- **Fix the labels.** On this dataset, cleaning the glacier/mountain boundary is probably  worth more than any architecture change.Same model from the command line, once trained:```bashpython scripts/evaluate.py --save-errors     # metrics + a CSV of every mistakepython scripts/predict.py path/to/photo.jpg  # single imagepython app.py                                # browser demo```